[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn as nn
import math

/usr/local/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [3]:
# ✏️ YOUR IMPLEMENTATION HERE
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        # W_q, W_k, W_v, W_o
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.d_model = d_model
        self.num_heads = num_heads

    def forward(self, x_q, x_kv):
        # Q from x_q, K/V from x_kv, no causal mask
        batch_size, s_q, _ = x_q.shape
        batch_size, s_k, _ = x_kv.shape
        d_head = self.d_model // self.num_heads

        Q = self.W_q(x_q)
        K = self.W_k(x_kv)
        V = self.W_v(x_kv)

        Q = Q.view(batch_size, s_q, self.num_heads, d_head).transpose(1,2)
        K = K.view(batch_size, s_k, self.num_heads, d_head).transpose(1,2)
        V = V.view(batch_size, s_k, self.num_heads, d_head).transpose(1,2)

        context_vec = torch.softmax(Q @ K.transpose(-1,-2) / math.sqrt(d_head), dim = -1) @ V
        context_vec = context_vec.transpose(1,2).contiguous().view(batch_size, s_q, self.d_model)
        context_vec = self.W_o(context_vec)

        return context_vec

In [4]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (1.6ms)
  ✅ [2/4] Q and KV different lengths (0.9ms)
  ✅ [3/4] No causal mask — all KV affects all Q (17.8ms)
  ✅ [4/4] Gradient flow (10.8ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (31.1ms total)
  Progress saved. Run status() to see your dashboard.

